In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
import requests
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score


# Reuse a single HTTP Session to enable HTTP Keep-Alive (drastically speeds up connections)
session = requests.Session()
session.headers.update(
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
    }
)

def get_games_by_publisher_steam(publisher_name, max_pages=3):
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    catalog = []

    for page in range(1, max_pages + 1):
        url = f"https://store.steampowered.com/search/?publisher={requests.utils.quote(publisher_name)}&page={page}"
        response = requests.get(url, headers=headers)

        if response.status_code == 429:
            print(
                "Rate limited (429)! Waiting 10 seconds before retrying..."
            )  #
            time.sleep(10)  #
            continue

        if response.status_code != 200:
            print(f"Error fetching page {page}: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        game_rows = soup.find_all("a", class_="search_result_row")

        if not game_rows:
            break

        for row in game_rows:
            title_elem = row.find("span", class_="title")
            appid = row.get("data-ds-appid")

            if title_elem and appid:
                catalog.append(
                    {"title": title_elem.text.strip(), "steam_appid": str(appid)}
                )

        # Pause for 1.5 seconds between page requests to stay under Steam's rate limit
        time.sleep(1.5)

    df = pd.DataFrame(catalog)
    if not df.empty:
        df = df[~df["steam_appid"].str.contains(",")].copy()
        df = df.drop_duplicates(subset=["steam_appid"]).reset_index(drop=True)
        df = df[
            ~df["title"].str.contains(
                "DLC|VR|Soundtrack|Pack|Expansion|Season Pass", case=False
            )
        ].reset_index(drop=True)

    return df


def fetch_single_game_data(appid):
    """Worker function: Fetches app details and review count for a single AppID."""
    details_url = f"https://store.steampowered.com/api/appdetails?appids={appid}&filters=basic"

    try:
        # Step 1: Check game details
        res = session.get(details_url, timeout=5)
        if res.status_code == 200:
            json_data = res.json()
            if (
                json_data
                and str(appid) in json_data
                and json_data[str(appid)].get("success")
            ):
                app_data = json_data[str(appid)]["data"]
                app_type = app_data.get("type", "").lower()

                # Strictly process base games
                if app_type == "game":
                    # Step 2: Fetch review count
                    reviews_url = f"https://store.steampowered.com/appreviews/{appid}?json=1&purchase_type=all"
                    rev_res = session.get(reviews_url, timeout=5)
                    total_reviews = 0

                    if rev_res.status_code == 200:
                        rev_data = rev_res.json()
                        if rev_data.get("success") == 1:
                            total_reviews = (
                                rev_data.get("query_summary", {}).get(
                                    "total_reviews", 0
                                )
                            )

                    return {
                        "steam_appid": str(appid),
                        "title": app_data.get("name", ""),
                        "type": app_type,
                        "total_reviews": total_reviews,
                    }
    except Exception as e:
        pass

    return None


def get_verified_games_with_reviews_fast(appid_list, max_workers=6):
    """Uses ThreadPoolExecutor to inspect multiple AppIDs in parallel."""
    valid_games = []
    print(
        f"Inspecting {len(appid_list)} potential AppIDs using {max_workers} parallel workers..."
    )

    start_time = time.time()

    # Parallel worker execution
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_appid = {
            executor.submit(fetch_single_game_data, appid): appid
            for appid in appid_list
        }

        # Gather results as they complete
        for future in as_completed(future_to_appid):
            result = future.result()
            if result:
                valid_games.append(result)

    elapsed = time.time() - start_time
    print(f"Finished processing in {elapsed:.2f} seconds!")

    return pd.DataFrame(valid_games)

def filter_by_review_percentile(df_games, drop_percentile, min_catalog_size):
    """Filters out the bottom X percentile of games based on review count for the current publisher.

    drop_percentile = 0.10 drops the bottom 10%
    """
    if df_games.empty:
        return df_games

    total_games = len(df_games)

    # GUARD: If a publisher only has a handful of games, don't drop anything!
    if total_games <= min_catalog_size:
        print(
            f"\n--- Publisher Review Percentile Analysis ---\n"
            f"Catalog is small ({total_games} games <= {min_catalog_size}). Skipping percentile trim.\n"
        )
        return df_games.reset_index(drop=True)
        
    # Calculate the review count threshold for the given percentile (e.g. 10th percentile)
    cutoff_threshold = df_games["total_reviews"].quantile(drop_percentile)

    print(f"\n--- Publisher Review Percentile Analysis ---")
    print(f"Total base games evaluated: {len(df_games)}")
    print(f"Calculated {int(drop_percentile * 100)}th percentile cutoff: {cutoff_threshold:.1f} reviews")

    # Keep games above or equal to the threshold
    filtered_df = df_games[
        df_games["total_reviews"] >= cutoff_threshold
    ].reset_index(drop=True)

    print(f"Remaining games after dropping bottom {int(drop_percentile * 100)}%: {len(filtered_df)}")
    return filtered_df

def publisher_catalog_dict(publisher, drop_percentile=0.10, min_catalog_size=15):
    # 1. Scrape raw publisher AppIDs
    raw_publisher_df = get_games_by_publisher_steam(
        publisher, max_pages=30
    )
    appid_list = raw_publisher_df["steam_appid"].tolist()
    
    # 2. Get verified base games and review totals (Fast multithreaded call)
    catalog_df = get_verified_games_with_reviews_fast(appid_list, max_workers=6)
    
    # 3. Apply your small-catalog aware percentile filter
    final_catalog_df = filter_by_review_percentile(
        catalog_df, drop_percentile, min_catalog_size
    )
    
    # print("\n--- Final Clean Catalog ---")
    # print(final_catalog_df.sort_values(by="total_reviews", ascending=True))
    final_catalog_dict=pd.Series(final_catalog_df["title"].values, index=final_catalog_df["steam_appid"]).to_dict()
    return final_catalog_dict


    
######################################################################################################



def get_game_price_history_itad(steam_appid, api_key, since_date="2015-01-01T00:00:00Z"):
    """Fetches full historical price log for a Steam AppID via ITAD API v2."""
    # Step 1: Lookup ITAD internal Game ID
    lookup_url = "https://api.isthereanydeal.com/games/lookup/v1"
    lookup_res = requests.get(
        lookup_url, params={"key": api_key, "appid": steam_appid}
    ).json()

    game_id = lookup_res.get("game", {}).get("id")
    if not game_id:
        print(f"Could not find ITAD game ID for Steam AppID {steam_appid}")
        return pd.DataFrame()

    # Step 2: Fetch history log for Steam (shop ID = 61)
    history_url = "https://api.isthereanydeal.com/games/history/v2"
    params = {
        "key": api_key,
        "id": game_id,
        "shops": 61,
        "since": since_date  # ISO 8601 string format
    }

    history_res = requests.get(history_url, params=params)

    if history_res.status_code != 200:
        print(f"Error fetching history: {history_res.status_code}")
        return pd.DataFrame()

    history_data = history_res.json()

    # Step 3: Parse history records cleanly
    records = []
    for entry in history_data:
        deal = entry.get("deal", {})
        records.append(
            {
                "timestamp_raw": entry.get("timestamp"),
                "date": pd.to_datetime(entry.get("timestamp")),
                "price": deal.get("price", {}).get("amount"),
                "regular_price": deal.get("regular", {}).get("amount"),
                "cut": deal.get("cut", 0),  # Discount percentage
            }
        )

    df = pd.DataFrame(records)

    # Sort chronologically from oldest to newest
    if not df.empty:
        df = df.sort_values(by="date").reset_index(drop=True)

    return df

def convert_itad_to_target_format(df_itad):
    """Converts ITAD price history DataFrame to match the target schema:

    DateTime | Final price | Historical Low
    """
    if df_itad.empty:
        return pd.DataFrame(
            columns=["DateTime", "Final price", "Retail Price", "Historical Low"]
        )

        
    df = df_itad.copy()

    # 1. Parse date with utc=True to handle mixed timezone offsets safely
    df["DateTime"] = (
        pd.to_datetime(df["timestamp_raw"], utc=True)
        .dt.tz_convert(None)  # Strips timezone after converting to UTC
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

    # 2. Map Final price
    df=df.rename(columns={"price":"Final price", "regular_price":"Retail Price"})

    # 3. Calculate running Historical Low (cumulative minimum up to each row)
    df["Historical Low"] = df["Final price"].cummin()

    # 4. Select and return target columns
    df=df[["DateTime","Final price","Historical Low","Retail Price"]]

    return df
    
ITAD_API_KEY = "0c3929a700f88f22c3ae0b877c3fa41fcf3720c7"

def generate_game_data(steam_appid):
    return convert_itad_to_target_format(get_game_price_history_itad(steam_appid, ITAD_API_KEY, since_date="2000-01-01T00:00:00Z"))


######################################################################################################


def initial_game_processing(df, game_title):

    df=df.copy()
    df["Game"]=game_title
    df["Game"]=df["Game"].astype("category")
    df=df.rename(columns={"Final price":"Current Price","DateTime":"Date"})
    df["Current Price"]=df["Current Price"].replace(0, np.nan)
    df=df.dropna(how="any").reset_index(drop=True)
    df["Date"]=pd.to_datetime(df["Date"])
    df=df[["Game","Date","Current Price","Historical Low","Retail Price"]].sort_values(by=["Game","Date"])
    df_next=df.shift(-1)
    # df["Retail Price"] =np.maximum(df_next["Current Price"], df["Current Price"])
    discounts=df[df["Current Price"] < df["Retail Price"]]
    discounts["Days Since Discount"]=discounts["Date"].diff(periods=1).dt.days
    discounts["Month"]=discounts["Date"].dt.month
    discounts["DayofWeek"]=discounts["Date"].dt.dayofweek
    discounts["DayofYear"]=discounts["Date"].dt.dayofyear
    discounts["Occurrence"] = np.ceil(discounts["Date"].dt.day / 7).astype(int)
    discounts["Days Until Next Sale"]=discounts["Days Since Discount"].shift(-1)
    discounts=discounts.dropna(subset=["Days Until Next Sale", "Days Since Discount"])
    return df, discounts


def prediction_function(discounts_list):
    discounts=pd.concat(discounts_list, ignore_index=True)
    discounts=discounts[discounts["Days Since Discount"]>0]
    
    
    X = discounts.drop(
        columns=["Days Since Discount","Current Price","Game","Days Until Next Sale", "Date", "Last_ATL_Date"], errors="ignore"
    )
    Y = discounts["Days Until Next Sale"]
    
    
    
    X_train, X_test, Y_train, Y_test, = train_test_split(X,Y,test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42)
    model.fit(X_train, Y_train)
    predictions = model.predict(X_test)
    
    
    importance_df = pd.DataFrame(
        {"Feature": X.columns, "Importance": model.feature_importances_}
    ).sort_values("Importance", ascending=False)
    
    print(importance_df)
    
    
    mae=mean_absolute_error(Y_test,predictions)
    r2=r2_score(Y_test,predictions)
    info=f"Mean Absolute Error: {mae:.2f} days\nR^2 Score: {r2:.2f}"
    comparison=pd.DataFrame({
        "Actual Days Until Next Sale":Y_test.values.ravel(),
        "Predicted Days Until Next Sale":predictions.round(1)
    }).reset_index(drop=True)
    return info,comparison


publisher="Bethesda Softworks"
# catalog_dict=publisher_catalog_dict(publisher, drop_percentile=0.25, min_catalog_size=30)
# catalog_dict={"379720":"DOOM"}
game_files_list=[]
discounts_list=[]
for appid, title in catalog_dict.items():
    raw_df=generate_game_data(appid)
    processed_df, discount_df=initial_game_processing(raw_df,title)
    if not processed_df.empty:
        game_files_list.append(processed_df)
    if not discount_df.empty:
        discounts_list.append(discount_df)

info1,comparison1=prediction_function(discounts_list)
print(info1)
print(comparison1)

          Feature  Importance
4       DayofYear    0.661338
2           Month    0.222325
0  Historical Low    0.072959
3       DayofWeek    0.022884
5      Occurrence    0.010902
1    Retail Price    0.009593
Mean Absolute Error: 16.05 days
R^2 Score: 0.60
    Actual Days Until Next Sale  Predicted Days Until Next Sale
0                           6.0                            19.1
1                          57.0                            67.1
2                          37.0                            65.5
3                          35.0                            38.4
4                         106.0                            67.1
5                          12.0                            18.2
6                          65.0                            65.3
7                          17.0                            17.7
8                          21.0                            17.7
9                          14.0                            18.2
10                         53.0       